# Petit tour des salaires

On explore uniquement les données du secteur privé : salaires par secteur, sexe, taille d’entreprise et temps de travail, puis quelques indicateurs d’inégalités et leur évolution.

**Avant de commencer :** lance d’abord `Nettoyage_donnees.ipynb`. Ce notebook lit les CSV dans `Data nettoyees/` et ne modifie aucun fichier.

## 1. On charge les données

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def trouver_racine_projet():
    courant = Path.cwd().resolve()
    for dossier in (courant, *courant.parents):
        if (dossier / 'Data nettoyees').is_dir():
            return dossier
    raise FileNotFoundError(
        'Dossier « Data nettoyees » introuvable. Lance d’abord le notebook de nettoyage.'
    )


RACINE = trouver_racine_projet()
DOSSIER_DONNEES = RACINE / 'Data nettoyees'
CHEMIN_PRIVE = DOSSIER_DONNEES / 'DS_DERA_PRIVE_SERIES_LONGUES_data_nettoye.csv'
if not CHEMIN_PRIVE.is_file():
    raise FileNotFoundError(
        f'Fichier privé nettoyé absent : {CHEMIN_PRIVE}. '
        'Lance d’abord le notebook de nettoyage.'
    )

prive = pd.read_csv(CHEMIN_PRIVE, sep=';', encoding='utf-8-sig')
print(f"Données privées : {len(prive):,} lignes, de {prive['annee'].min()} à {prive['annee'].max()}.")

## 2. Quelques repères avant les graphiques

Les montants sont en **euros constants**, donc corrigés de l’inflation. Pour comparer une dimension (par exemple le sexe), on garde les autres dimensions sur « Ensemble ». Ça évite de comparer des groupes qui ne correspondent pas.

Les salaires sont des moyennes ou des quantiles publiés, pas des observations individuelles.

In [ ]:
SALAIRE = 'Salaire net moyen mensuel (Euros constants)'
MOYENNE = 'Ensemble'
P10 = '10e centile (D1 - 10% gagnent moins)'
P25 = '1er quartile (Q1 - 25% gagnent moins)'
P50 = 'Médiane (50% gagnent moins/plus)'
P90 = '90e centile (D9 - 10% gagnent plus)'
P95 = '95e centile (Top 5%)'
P99 = '99e centile (Top 1%)'

VALEURS_ENSEMBLE = {
    'secteur_activite': 'Ensemble des secteurs',
    'tranche_age': 'Ensemble',
    'taille_entreprise': 'Ensemble',
    'categorie_socio_prof': 'Ensemble',
    'sexe': 'Ensemble',
    'temps_travail': 'Ensemble',
}


def garder_ensemble_des_autres_dimensions(df, dimension=None):
    resultat = df.copy()
    for colonne, modalite in VALEURS_ENSEMBLE.items():
        if colonne in resultat.columns and colonne != dimension:
            resultat = resultat[resultat[colonne].eq(modalite)]
    return resultat


def extraire_salaire(df, quantiles, dimension=None):
    resultat = garder_ensemble_des_autres_dimensions(df, dimension)
    resultat = resultat[
        resultat['indicateur'].eq(SALAIRE)
        & resultat['unite'].eq('Euros constants')
        & resultat['centile_salaire'].isin(quantiles)
    ].copy()
    cles = ['annee', 'centile_salaire']
    if dimension:
        cles.insert(0, dimension)
    if resultat.duplicated(cles).any():
        raise ValueError(f'Plusieurs valeurs pour les mêmes clés : {cles}')
    return resultat


def derniere_annee_complete(table, groupe):
    annees_max = table.groupby(groupe)['annee'].max()
    return int(annees_max.min())


def tableau_salaire_recent(df, dimension):
    table = extraire_salaire(df, [MOYENNE], dimension)
    annee = derniere_annee_complete(table, dimension)
    table = table[table['annee'].eq(annee)].copy()
    table = table.rename(columns={dimension: 'Groupe', 'valeur': 'Salaire moyen (€ constants)'})
    return table[['Groupe', 'Salaire moyen (€ constants)']].sort_values(
        'Salaire moyen (€ constants)', ascending=False
    ), annee

## 3. Les salaires selon les groupes

Pour chaque tableau, l’année de référence est la dernière année disponible pour tous les groupes comparés. « Ensemble » sert de repère quand il est présent.

In [ ]:
comparaisons = [
    ('secteur_activite', 'Par secteur'),
    ('sexe', 'Par sexe'),
    ('taille_entreprise', 'Par taille d’entreprise'),
    ('temps_travail', 'Par temps de travail'),
]

tables_groupes = {}
for dimension, titre in comparaisons:
    table, annee = tableau_salaire_recent(prive, dimension)
    tables_groupes[titre] = table
    print(f'{titre} — {annee}')
    display(table.reset_index(drop=True).style.format({'Salaire moyen (€ constants)': '{:,.0f}'}))

secteurs = tables_groupes['Par secteur']
secteurs = secteurs[secteurs['Groupe'].ne('Ensemble des secteurs')]
secteurs.sort_values('Salaire moyen (€ constants)').plot.barh(
    x='Groupe', y='Salaire moyen (€ constants)', legend=False, figsize=(10, 7)
)
plt.title('Salaire moyen par secteur — privé')
plt.xlabel('Euros constants par mois')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 4. Un ratio pour regarder les écarts

Ici, le ratio **P90 / P10** compare le salaire au 90e centile à celui du 10e. Par exemple, un ratio de 3 signifie que le P90 est trois fois le P10. C’est un indicateur simple de dispersion, pas une mesure complète des inégalités.

Les quantiles ne sont pas publiés pour chaque découpage. On affiche donc les comparaisons par groupe seulement quand les deux valeurs (P10 et P90) sont bien présentes.

In [ ]:
def calculer_ratio_90_10(df, dimension=None):
    table = extraire_salaire(df, [P10, P90], dimension)
    index = ['annee'] if dimension is None else [dimension, 'annee']
    large = table.pivot(index=index, columns='centile_salaire', values='valeur')
    large = large.dropna(subset=[P10, P90]).copy()
    large['Ratio P90/P10'] = large[P90] / large[P10]
    return large.reset_index()


ratios = calculer_ratio_90_10(prive)
display(ratios[['annee', 'Ratio P90/P10']].tail(10).style.format({'Ratio P90/P10': '{:.2f}'}))
ratios.plot(x='annee', y='Ratio P90/P10', legend=False, figsize=(9, 4))
plt.title('Évolution du ratio P90/P10 — privé')
plt.xlabel('Année')
plt.ylabel('Ratio')
plt.tight_layout()
plt.show()

for dimension, titre in [('sexe', 'Par sexe'), ('temps_travail', 'Par temps de travail')]:
    ratios_groupe = calculer_ratio_90_10(prive, dimension)
    groupes = ratios_groupe[ratios_groupe[dimension].ne('Ensemble')]
    annee = derniere_annee_complete(groupes, dimension)
    resultat = groupes[groupes['annee'].eq(annee)].sort_values(
        'Ratio P90/P10', ascending=False
    )
    print(f'{titre} — {annee}')
    display(
        resultat[[dimension, 'Ratio P90/P10']]
        .reset_index(drop=True)
        .style.format({'Ratio P90/P10': '{:.2f}'})
    )

print('Les quantiles P10 et P90 ne sont pas publiés par secteur ni par taille d’entreprise.')

## 5. On regarde le haut de la distribution

On suit les salaires P90, P95 et P99 en euros constants.

In [ ]:
def serie_quantile(df, quantile, etiquette):
    table = extraire_salaire(df, [quantile])
    table = table[['annee', 'valeur']].rename(columns={'valeur': 'Salaire (€ constants)'})
    table['Série'] = etiquette
    return table


series_hauts = pd.concat(
    [
        serie_quantile(prive, P90, 'P90'),
        serie_quantile(prive, P95, 'P95'),
        serie_quantile(prive, P99, 'P99'),
    ],
    ignore_index=True,
)

pivot_hauts = series_hauts.pivot(index='annee', columns='Série', values='Salaire (€ constants)')
display(pivot_hauts.tail(10).style.format('{:,.0f}'))
pivot_hauts.plot(figsize=(10, 5))
plt.title('Évolution des hauts salaires — privé')
plt.xlabel('Année')
plt.ylabel('Euros constants par mois')
plt.tight_layout()
plt.show()

def bilan_evolution(series):
    bilans = []
    for nom, groupe in series.groupby('Série'):
        groupe = groupe.sort_values('annee')
        debut, fin = groupe.iloc[0], groupe.iloc[-1]
        variation = (fin['Salaire (€ constants)'] / debut['Salaire (€ constants)'] - 1) * 100
        bilans.append(
            {
                'Série': nom,
                'Période': f"{int(debut['annee'])}–{int(fin['annee'])}",
                'Début (€)': debut['Salaire (€ constants)'],
                'Fin (€)': fin['Salaire (€ constants)'],
                'Variation (%)': variation,
            }
        )
    return pd.DataFrame(bilans).set_index('Série')


display(bilan_evolution(series_hauts).style.format({'Début (€)': '{:,.0f}', 'Fin (€)': '{:,.0f}', 'Variation (%)': '{:+.1f}%'}))

### Quelle part de la masse salariale va aux plus hauts revenus ?

Le fichier privé contient aussi la part de la masse salariale reçue par les 10 %, 1 %, 0,1 % et 0,01 % les mieux rémunérés. C’est un autre indicateur que P90/P10 : on suit ici une **part en pourcentage**, pas un salaire.

In [ ]:
part_top = garder_ensemble_des_autres_dimensions(prive)
part_top = part_top[
    part_top['indicateur'].str.startswith('Part de la masse salariale perçue par le top ')
    & part_top['centile_salaire'].eq('Ensemble')
].copy()
part_top['Groupe'] = part_top['indicateur'].str.extract(r'(top .+)$', expand=False)
part_top = part_top.pivot(index='annee', columns='Groupe', values='valeur')
display(part_top.tail(10).style.format('{:.2f}%'))
part_top.plot(figsize=(10, 5))
plt.title('Part de la masse salariale détenue par le haut de la distribution — privé')
plt.xlabel('Année')
plt.ylabel('Part (%)')
plt.tight_layout()
plt.show()

## 6. Et du côté des salaires les plus bas ?

On suit P10 et P25 avec la médiane comme repère. Comme pour les salaires du haut, on compare les euros constants et on affiche les dates réellement disponibles.

In [ ]:
series_bas = pd.concat(
    [
        serie_quantile(prive, P10, 'P10'),
        serie_quantile(prive, P25, 'P25'),
        serie_quantile(prive, P50, 'Médiane'),
    ],
    ignore_index=True,
)

pivot_bas = series_bas.pivot(index='annee', columns='Série', values='Salaire (€ constants)')
display(pivot_bas.tail(10).style.format('{:,.0f}'))
pivot_bas.plot(figsize=(10, 5))
plt.title('Évolution des bas salaires — privé')
plt.xlabel('Année')
plt.ylabel('Euros constants par mois')
plt.tight_layout()
plt.show()

display(bilan_evolution(series_bas).style.format({'Début (€)': '{:,.0f}', 'Fin (€)': '{:,.0f}', 'Variation (%)': '{:+.1f}%'}))

## À garder en tête

- Les quantiles P10 et P90 ne sont pas détaillés par secteur ni par taille d’entreprise, donc on ne peut pas calculer les ratios d’inégalité pour ces groupes.
- Les périodes ne sont pas toujours les mêmes et certaines années peuvent manquer. Les tableaux d’évolution indiquent les années de début et de fin effectivement observées.
- Le ratio P90/P10 résume un écart entre deux points de la distribution; il ne raconte pas toute l’histoire des inégalités.
- Ces données décrivent des salaires mensuels en équivalent temps plein (EQTP); elles ne mesurent pas directement le revenu annuel ou le niveau de vie.